# Tutorial 03 - Project 1: Modular Sales Inventory Processing

This notebook processes raw inventory lines, instantiates object-oriented
`Product` entities, and plots the total inventory valuation.


---
## Step 0: Portable Environment Bootstrap (Local IDE / Jupyter + Google Colab)

Locates the `APDS/02_Sales_Inventory_Pipeline/` project folder (anchored on `data/raw/data.txt` — it also puts the project on `sys.path` so `from utils import ...` works from anywhere) by walking upward from the
current working directory — so the notebook runs identically from the repository
root, the project folder or its `notebooks/` sub-folder. On Google Colab the
repository is cloned into `/content/APSD_TAI_LAB` automatically (opening the
notebook directly from GitHub also works). The resolved project folder is added
to `sys.path` so its local modules (e.g. `utils`, `students`) import cleanly.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_NAME = "02_Sales_Inventory_Pipeline"
ANCHOR_REL = "data/raw/data.txt"  # file that uniquely identifies the project folder
REPO_URL = "https://github.com/Vtheonly/APSD_TAI_LAB.git"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    # Google Colab: clone the repository (unless already present, e.g. on Drive)
    _repo = Path("/content/APSD_TAI_LAB")
    if not (_repo / "APDS").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True
        )
    PROJ_DIR = _repo / "APDS" / PROJECT_NAME
else:
    # Local IDE / Jupyter: find the project folder by walking upward from the
    # working directory (works from the repo root, the project folder or its
    # notebooks/ sub-folder, regardless of where Jupyter was launched).
    _cwd = Path.cwd().resolve()
    PROJ_DIR = next(
        (
            c / "APDS" / PROJECT_NAME
            for c in (_cwd, *_cwd.parents)
            if (c / "APDS" / PROJECT_NAME / ANCHOR_REL).exists()
        ),
        None,
    )
    if PROJ_DIR is None:
        raise FileNotFoundError(
            f"Project folder APDS/{PROJECT_NAME} (anchor: {ANCHOR_REL}) not found "
            f"above {_cwd}. Launch Jupyter from inside the cloned repository."
        )
if str(PROJ_DIR) not in sys.path:
    sys.path.insert(0, str(PROJ_DIR))

print(f"Project folder : {PROJ_DIR}")
print(f"Google Colab   : {IN_COLAB}")


### 1. Read the Raw Text Data


In [ ]:
from utils import clean_lines, clean_lines_loop, parse_products, Product
import matplotlib.pyplot as plt

with open(PROJ_DIR / "data" / "raw" / "data.txt", "r", encoding="utf-8") as f:
    raw_lines = f.readlines()

print(f"Total Raw Lines: {len(raw_lines)}")


### 2. Compare Loop vs. List Comprehension Cleaning


In [ ]:
cleaned_loop = clean_lines_loop(raw_lines)
cleaned_comp = clean_lines(raw_lines)

assert cleaned_loop == cleaned_comp
print(f"Cleaned Lines ({len(cleaned_comp)} records):")
for line in cleaned_comp:
    print(f"  {line}")


### 3. Parse Records and Instantiate Product Objects


In [ ]:
parsed_tuples = parse_products(cleaned_comp)
products = [Product(name, price, qty) for name, price, qty in parsed_tuples]

for p in products:
    print(f"{p} -> Inventory Value: ${p.total_value():,.2f}")


### 4. Valuation Bar Chart


In [ ]:
names = [p.name for p in products]
values = [p.total_value() for p in products]

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(names, values, color="steelblue", edgecolor="black", alpha=0.85)

ax.set_title("Total Sales Inventory Valuation by Product", fontsize=13, fontweight="bold", pad=10)
ax.set_ylabel("Total Value ($)", fontsize=11)
ax.grid(axis="y", linestyle="--", alpha=0.5)

for bar in bars:
    y = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, y + 50, f"${y:,.0f}",
            ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()


### 5. The `Product.__str__` Showcase

Every print of a `Product` goes through its `__str__` method — the
human-readable representation defined in `utils/product.py`. Printing the
object directly, calling `str()` on it, and interpolating it into an f-string
all produce the same formatted line, without ever calling `total_value()`
by hand inside the format string.

In [ ]:
# `print(product)` calls Product.__str__ under the hood — same line as
# `str(product)` and as f-string interpolation.
first, second = products[0], products[1]

print(first)
print(str(second))
print(f"Cheapest to reprice: {second}")

# Contrast: a plain class without __str__ falls back to the default
# <module.Class object at 0x...> repr — that is what Product overrides.
class WithoutStr:
    def __init__(self, name):
        self.name = name

plain = WithoutStr("Laptop")
print(plain)

# Re-read what the formatted string actually contains.
assert str(first) == f"Product({first.name}, Price={first.price:.2f}, Qty={first.quantity})"
print("OK — str(product) matches the f-string built from the same fields.")

### 6. Save the Processed Data under `data/processed/`

The pipeline does not stop at the chart: the parsed records are persisted as
a processed-data artifact, so later steps (or the streaming exercise in
`APDS/04`) can start from clean data instead of re-parsing the raw text.
`save_processed_data` (in `utils/helpers.py`) writes the cleaned lines and a
CSV summary that includes the computed `total_value` column.

In [ ]:
from utils import save_processed_data

processed_dir = PROJ_DIR / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

txt_path, csv_path = save_processed_data(parsed_tuples, processed_dir)
print(f"Cleaned data written to: {txt_path.relative_to(PROJ_DIR)}")
print(f"CSV summary written to: {csv_path.relative_to(PROJ_DIR)}")

# Round-trip check: the CSV carries every parsed record plus its total value.
with open(csv_path, "r", encoding="utf-8") as f:
    rows = f.read().splitlines()

header, *data_rows = rows
assert header == "name,price,quantity,total_value"
assert len(data_rows) == len(products)
for row, prod in zip(data_rows, products):
    name, price, qty, total = row.split(",")
    assert (name, float(price), int(qty), float(total)) == (
        prod.name, prod.price, prod.quantity, prod.total_value()
    )
print("Round-trip OK — processed CSV matches the in-memory products.")